# Chip-Level Test Time Optimizer: Public Evidence

This notebook walks through the frozen public synthetic Project 05 evidence. It does not generate random predictions or estimate production savings.

**Accepted boundary:** 13.524% simulated optional-stage reduction, 99.0% defect recall, six escaped failures among 600, and 4.149% over-test on 10,000 new chronological synthetic chips. The 15% reduction and zero-observed-escape objectives remain unmet.

In [6]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "evidence").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "evidence").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from deployment.runtime import HybridPolicyRuntime
from scripts.validate_evidence import validate


def read_json(relative_path: str) -> dict:
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))


public = read_json("evidence/public_synthetic_evaluation.json")
dataset = read_json("evidence/public_synthetic_dataset_manifest.json")
operational = read_json("evidence/operational_envelope_confirmation.json")
claims = read_json("evidence/claims.json")

print("Repository:", ROOT.name)
print("Data scope:", claims["data_scope"]["provenance"])
print("Bundle:", claims["champion"]["bundle_id"])
print("Benchmark splits:", len(dataset["splits"]))

Repository: 05_Chip_Level_Test_Time_Optimizer
Data scope: independently generated synthetic data; no external or confidential dataset
Bundle: 53ce0e9ccbd63b3c84c581a0dedc325782e8c09b72847977626f41aa6ad3d1fe
Benchmark splits: 8


## Data Roles And Isolation

The benchmark separates fitting, known-mode policy selection, OOD drift calibration, and frozen confirmation by lot and chronological range. A later post-freeze confirmation uses 20 additional lots. All rows are independently generated synthetic data.

In [2]:
split_rows = [
    {
        "split": name,
        "chips": details["samples"],
        "lots": details["lot_count"],
        "failures": details["fail_samples"],
        "time_min": details["time_index_min"],
        "time_max": details["time_index_max"],
    }
    for name, details in dataset["splits"].items()
]
split_table = pd.DataFrame(split_rows).sort_values("time_min")
display(split_table)

assert dataset["isolation"] == {
    "chip_overlap": 0,
    "lot_overlap": 0,
    "time_overlap": 0,
}
print("Benchmark chips:", int(split_table["chips"].sum()))
print("Post-freeze chips:", operational["metrics"]["total_chips"])
print("Feature count:", dataset["feature_count"])
print("Base seed:", dataset["base_seed"])

,split,chips,lots,failures,time_min,time_max
6,train,16000,32,960,0,31
7,validation,4000,8,240,40,47
1,known_shift_validation,6000,12,360,60,71
4,ood_validation,2000,4,400,80,83
2,known_shift_validation_2,10000,20,600,100,119
5,ood_validation_2,5000,10,1000,130,139
0,confirmation,10000,20,600,160,179
3,ood_confirmation,5000,10,1000,190,199


Benchmark chips: 58000
Post-freeze chips: 10000
Feature count: 32
Base seed: 20260807


## Bounded Candidate Selection

The catalog contains 198 measured candidates. Only 180 hybrid `classifier OR VAE OR sigma` policies are eligible. Selection maximizes validation simulated reduction subject to limits on observed escapes, relative escape rate, one-sided uncertainty, and over-test. Confirmation data is not used for selection.

In [3]:
selected = public["validation"]["selected_candidate"]
comparison_names = {
    "all_test_baseline",
    "classifier_t0.20",
    "vae_q990",
    "sigma_tail_0.00010",
    selected["name"],
}
comparison_rows = []
for candidate in public["validation"]["candidates"]:
    if candidate["name"] in comparison_names:
        metrics = candidate["metrics"]
        comparison_rows.append(
            {
                "candidate": candidate["name"],
                "family": candidate["policy_family"],
                "eligible": candidate["eligible_for_selection"],
                "reduction_percent": metrics["time_reduction_percent"],
                "relative_escape_rate": metrics["relative_escape_rate"],
                "overtest_rate": metrics["overtest_rate"],
                "gate_passed": candidate["gate_result"]["passed"],
            }
        )
comparison = pd.DataFrame(comparison_rows)
display(comparison)

assert public["validation"]["candidate_count"] == 198
assert public["validation"]["eligible_candidate_count"] == 180
print("Selected:", selected["name"])
print("Objective:", public["selection_protocol"]["objective"])
print("Gates:", public["selection_protocol"]["safety_gates"])

,candidate,family,eligible,reduction_percent,relative_escape_rate,overtest_rate,gate_passed
0,all_test_baseline,baseline,False,0.00000,0.000000,1.000000,False
1,classifier_t0.20,classifier_ablation,False,13.43475,0.010833,0.047872,False
2,vae_q990,vae_ablation,False,13.81800,0.267500,0.037074,False
3,sigma_tail_0.00010,sigma_ablation,False,13.87575,0.365000,0.039202,False
4,hybrid_c0.20_vae_q990_sigma_tail_0.00010,hybrid_or,True,13.10850,0.008333,0.070851,True


Selected: hybrid_c0.20_vae_q990_sigma_tail_0.00010
Objective: maximize validation time reduction subject to every safety gate
Gates: {'maximum_escape_rate_upper_95': 0.02, 'maximum_observed_escapees': 10, 'maximum_overtest_rate': 0.25, 'maximum_relative_escape_rate': 0.01}


## Retained Failures And Accepted Post-Freeze Result

Two predecessor policies failed robustness. The selected policy's first frozen confirmation then blocked all 20 lots, yielding zero escapes but 100% over-test and 0% reduction. The later accepted result uses 20 new chronological lots inside the declared development envelope and performs no tuning or reselection.

In [4]:
failed_trials = pd.DataFrame(
    [
        {
            "trial": "Predecessor 1",
            "known_mode": "13.163% reduction; 2 escapes",
            "robustness": "76.5% OOD recall",
            "decision": "Rejected",
        },
        {
            "trial": "Predecessor 2",
            "known_mode": "10.791% reduction; 1 escape",
            "robustness": "96.5% OOD recall; 35 escapes",
            "decision": "Rejected",
        },
        {
            "trial": "First frozen selected-policy confirmation",
            "known_mode": "0% reduction; all 20 lots blocked",
            "robustness": "All ten OOD lots blocked",
            "decision": "Rejected on utility",
        },
    ]
)
display(failed_trials)

accepted = operational["metrics"]
accepted_table = pd.DataFrame(
    [
        ("Chips / failures", f"{accepted['total_chips']:,} / {accepted['failed_chips']:,}"),
        ("Simulated reduction", f"{accepted['time_reduction_percent']:.3f}%"),
        ("Reduction 95% interval", "13.434% to 13.609%"),
        ("Defect recall", f"{accepted['defect_recall']:.1%}"),
        ("Escapes", f"{accepted['escapees']} / {accepted['failed_chips']}"),
        ("One-sided escape upper 95%", f"{accepted['escape_rate_upper_95']:.3%}"),
        ("Over-test", f"{accepted['overtest_rate']:.3%}"),
        ("MCC", f"{accepted['mcc']:.4f}"),
    ],
    columns=["metric", "accepted_value"],
)
display(accepted_table)

failure_modes = pd.DataFrame.from_dict(
    operational["failure_modes"], orient="index"
).sort_values("recall")
display(failure_modes)
assert failure_modes.index[0] == "timing_shift"
assert accepted["time_reduction_percent"] == 13.524
assert accepted["escapees"] == 6

,trial,known_mode,robustness,decision
0,Predecessor 1,13.163% reduction; 2 escapes,76.5% OOD recall,Rejected
1,Predecessor 2,10.791% reduction; 1 escape,96.5% OOD recall; 35 escapes,Rejected
2,First frozen selected-policy confirmation,0% reduction; all 20 lots blocked,All ten OOD lots blocked,Rejected on utility


,metric,accepted_value
0,Chips / failures,"10,000 / 600"
1,Simulated reduction,13.524%
2,Reduction 95% interval,13.434% to 13.609%
3,Defect recall,99.0%
4,Escapes,6 / 600
5,One-sided escape upper 95%,1.964%
6,Over-test,4.149%
7,MCC,0.7563


,captured,escaped,recall,support
timing_shift,138,2,0.985714,140
leakage_spike,158,2,0.987500,160
voltage_drift,158,2,0.987500,160
resistance_bridge,140,0,1.000000,140


## Live Frozen Inference And Exact Evidence Replay

The public input is one unlabelled 500-chip lot from the accepted operational envelope. Runtime loading verifies model, evidence, dataset, schema, and bundle identities. The final replay regenerates all eight split hashes and the 10,000-chip operational prediction SHA-256.

In [5]:
runtime = HybridPolicyRuntime(ROOT / "artifacts/public_v1/runtime_manifest.json")
fixture = read_json("examples/public_synthetic_input.json")
fixture_frame = pd.DataFrame.from_records(fixture["records"])
predictions = runtime.predict_dataframe(fixture_frame)

skipped = int((predictions["flag"] == 0).sum())
run = len(predictions) - skipped
live_reduction = skipped / len(predictions) * 15 / (85 + 15) * 100
live_summary = pd.DataFrame(
    [
        ("Chips", len(predictions)),
        ("SKIP", skipped),
        ("RUN", run),
        ("Simulated reduction (%)", live_reduction),
        ("Drift-blocked chips", int(predictions["lot_drift_blocked"].sum())),
    ],
    columns=["live_metric", "value"],
)
display(live_summary)
print("Model:", runtime.config["model_id"])
print("Bundle:", runtime.manifest["bundle_id"])
assert skipped == 450
assert run == 50
assert live_reduction == 13.5

report = validate(ROOT)
replay_summary = {
    "status": report["status"],
    "split_hashes_passed": sum(report["dataset_split_checks"].values()),
    "claim_checks_passed": sum(report["claim_checks"].values()),
    "operational_metrics_replayed": report["operational_metrics_replayed"],
    "prediction_sha256": report["operational_prediction_sha256"],
}
print(json.dumps(replay_summary, indent=2))
assert report["status"] == "passed", report["errors"]
assert all(report["dataset_split_checks"].values())
assert all(report["claim_checks"].values())

,live_metric,value
0,Chips,500.0
1,SKIP,450.0
2,RUN,50.0
3,Simulated reduction (%),13.5
4,Drift-blocked chips,0.0


Model: public_synthetic_hybrid_v1
Bundle: 53ce0e9ccbd63b3c84c581a0dedc325782e8c09b72847977626f41aa6ad3d1fe
{
  "status": "passed",
  "split_hashes_passed": 8,
  "claim_checks_passed": 4,
  "operational_metrics_replayed": true,
  "prediction_sha256": "d5d50071d10de5bf0dfb531b843c49567a42cb170b595cc1b990f4c2a30661db"
}


## Limitations And Promotion Boundary

- All accepted evidence is independently generated synthetic data.
- 13.524% is simulated optional-stage reduction under an 85/15 cost model, not production savings.
- Six of 600 synthetic known-mode failures escaped; OR logic does not guarantee zero escapes.
- Timing shift was weakest at 98.571% recall; these are generator modes, not physical-defect coverage claims.
- The 15% reduction and zero-observed-escape objectives remain unmet.
- OOD safety is lot-level full testing with 0% reduction, not per-chip novel-defect recognition.
- Physical-defect coverage, hardware timing, production service levels, and deployment approval are not proven.
- Promotion requires approved representative data, broader stress lots, measured ATE timing, shadow mode, monitoring, rollback, and manufacturing/product-quality review.

Notebook code cells were executed in order against the frozen bundle; the saved tables and replay identity are the evidence shown above.